In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import json

Gemini solution:

In [1]:
import pandas as pd
from datasets import load_dataset


In [4]:
sample_size = 30
seed = 42
dataset_configs = {
    'fineweb': {"path": "HuggingFaceFW/fineweb", "name": "sample-10BT", "split": "train"},
    'finepdfs': {"path": "HuggingFaceFW/finepdfs", "name": "eng_Latn", "split": "train"},
    'C4': {"path": "allenai/c4", "name": "en", "split": "train"},
    'RedPajama': {"path": "krisbailey/RedPajama-Data-V2-1B", "name": None, "split": "train"}
}

In [6]:
for name, config in dataset_configs.items():
    print(f"Streaming {name}...")

    # 1. Load the streaming dataset
    kwargs = {"path": config["path"], "split": config["split"], "streaming": True}
    if config["name"]:
        kwargs["name"] = config["name"]

    ds = load_dataset(**kwargs)

    # 2. Increase buffer_size for a more representative shuffle over the stream
    # and take the sample size
    sampled_stream = ds.shuffle(seed=seed, buffer_size=10000).take(sample_size)

    # 3. Consume the stream into a list
    samples_list = list(sampled_stream)
    df = pd.DataFrame(samples_list)

    # 4. Identify or create the document index column for your assignment
    # Let's see what ID keys are available and standardize them into a 'document_id' column
    if 'id' in df.columns:
        df['document_index'] = df['id']
    elif 'doc_id' in df.columns:
        df['document_index'] = df['doc_id']
    elif 'url' in df.columns:
        # For C4, URL + timestamp is the standard unique identifier
        df['document_index'] = df['url']
    else:
        # Fallback if a dataset lacks an explicit ID string
        df['document_index'] = [f"generated_idx_{i}" for i in range(len(df))]

    # Save raw JSON for backup/inspection
    filename = f"{name}_dataset.json"
    df.to_json(filename, orient="records", indent=2)
    print(f"Saved {name} with ID columns: {[col for col in ['id', 'doc_id', 'url', 'document_index'] if col in df.columns]}")

print("\nReady for annotation setup!")

Streaming fineweb...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Saved fineweb with ID columns: ['id', 'url', 'document_index']
Streaming finepdfs...


Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

Saved finepdfs with ID columns: ['id', 'url', 'document_index']
Streaming C4...


...

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Saved C4 with ID columns: ['url', 'document_index']
Streaming RedPajama...


README.md:   0%|          | 0.00/2.21k [00:00<?, ?B/s]

Saved RedPajama with ID columns: ['doc_id', 'document_index']

Ready for annotation setup!


In [ ]:
fineweb_on_HF = "HuggingFaceFW/fineweb"
finepdfs_on_HF = "HuggingFaceFW/finepdfs"
C4_on_HF = "allenai/c4"
RedPajama_V2_sample = "togethercomputer/RedPajama-Data-V2"

datasources = [fineweb_on_HF, finepdfs_on_HF, C4_on_HF, RedPajama_V2_sample]

In [ ]:
sample_size = 30
datasets = {}
datasets['fineweb'] = load_dataset("HuggingFaceFW/fineweb", split="train", streaming=True).shuffle(seed=42).take(sample_size)
datasets['finepdfs'] = load_dataset("HuggingFaceFW/finepdfs", split="train", streaming=True).shuffle(seed=42).take(sample_size)
datasets['C4'] = load_dataset("allenai/c4", 'en', split='train', streaming=True).shuffle(seed=42).take(sample_size)
datasets['RedPajama'] = load_dataset("krisbailey/RedPajama-Data-V2-1B", split="train", streaming=True).shuffle(seed=42).take(sample_size)

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/230k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

In [ ]:
print("Overview of loaded datasets:")
for name, dataset in datasets.items():
    print(f"\nDataset Name: {name}")
    print(f"  Type: {type(dataset)}")
    if hasattr(dataset, 'features'):
        print(f"  Features: {list(dataset.features.keys())}")
    else:
        print("  Features: Not directly available or dataset is not fully materialized.")
    print(f"  Representation: {dataset.__repr__()[:150]}...") # Print a truncated representation

In [ ]:
for name, dataset in datasets.items():
    # Convert the IterableDataset to a list of dictionaries, then to a DataFrame
    df = pd.DataFrame(list(dataset))
    filename = f"{name}_dataset.json"
    df.to_json(filename, orient="records", indent=2)
    print(f"Saved {name} dataset to {filename}")

Saved fineweb dataset to fineweb_dataset.json
Saved finepdfs dataset to finepdfs_dataset.json
Saved C4 dataset to C4_dataset.json
Saved RedPajama dataset to RedPajama_dataset.json
